# Exploratory Data Analysis on Retail Sales Data

**Objective:** uncover sales patterns, customer behaviour trends, product/category performance, and actionable business insights from the retail sales dataset.

**Dataset used:** `Retail_Sales.csv`  
**Rows and columns:** 2,000 rows x 15 raw columns  
**Date range:** 2022-01-01 to 2023-12-31

This notebook uses Python, pandas, matplotlib, and seaborn. Each chart is followed by observations so the analysis reads like a business report as well as a technical notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (11, 6)
plt.rcParams["axes.titlesize"] = 15
plt.rcParams["axes.labelsize"] = 11

df = pd.read_csv("Retail_Sales.csv")
df.head()

## 1. Initial Dataset Inspection

The first step is to understand the dataset structure, data types, missing values, and a few sample records. The source file includes a typo in the quantity column name (`quantiy`), so the cleaning step below creates a clearer `quantity` alias while keeping the original column intact.

In [ ]:
print("Shape:", df.shape)
print("\nColumn data types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values by column:")
display(df.isna().sum().to_frame("missing_values"))

display(df.head())

**Observations**

- The dataset contains **2,000 rows and 15 raw columns**.
- Missing values are limited: age has **10** missing records, while quantity, price, cost, and total sale each have **3** missing records.
- `sale_date` and `sale_time` are stored as text, so they need to be converted before time series analysis.

In [ ]:
# Clean and enrich the dataset for analysis
retail = df.copy()
retail["sale_date"] = pd.to_datetime(retail["sale_date"], errors="coerce")
retail["sale_time"] = pd.to_datetime(retail["sale_time"], format="%H:%M:%S", errors="coerce").dt.time
retail["quantity"] = retail["quantiy"]
retail["month"] = retail["sale_date"].dt.to_period("M").dt.to_timestamp()
retail["quarter"] = retail["sale_date"].dt.to_period("Q").astype(str)
retail["hour"] = pd.to_datetime(retail["sale_time"], format="%H:%M:%S", errors="coerce").dt.hour

age_bins = [0, 24, 34, 44, 54, 64, 120]
age_labels = ["18-24", "25-34", "35-44", "45-54", "55-64", "65+"]
retail["age_group"] = pd.cut(retail["age"], bins=age_bins, labels=age_labels, right=True)

retail.head()

## 2. Descriptive Statistics

The table below reports the required mean, median, mode, and standard deviation for every numerical column. Transaction and customer IDs are included for completeness, although they are identifiers rather than business measures.

In [ ]:
numeric_cols = retail.select_dtypes(include="number").columns
summary_stats = retail[numeric_cols].agg(["mean", "median", "std"]).T
summary_stats["mode"] = retail[numeric_cols].mode(dropna=True).iloc[0]
summary_stats = summary_stats[["mean", "median", "mode", "std"]]
display(summary_stats.round(2))

**Observations**

- Average transaction revenue is about **456.54**, with a median of **150.00**, showing that larger baskets pull the mean upward.
- Average quantity per transaction is **2.51** units.
- Customer age centers around **41.3** years, with a broad spread across adult age bands.

## 3. Time Series Analysis

Monthly and quarterly sales trends reveal seasonality, campaign timing, and periods where demand changes materially.

In [ ]:
monthly_sales = retail.groupby("month", as_index=False)["total_sale"].sum()
quarterly_sales = retail.groupby("quarter", as_index=False)["total_sale"].sum()

fig, ax = plt.subplots(figsize=(12, 6))
sns.lineplot(data=monthly_sales, x="month", y="total_sale", marker="o", linewidth=2.5, ax=ax)
ax.set_title("Monthly Sales Trend")
ax.set_xlabel("Month")
ax.set_ylabel("Total Sales")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
sns.lineplot(data=quarterly_sales, x="quarter", y="total_sale", marker="o", linewidth=2.5, ax=ax)
ax.set_title("Quarterly Sales Trend")
ax.set_xlabel("Quarter")
ax.set_ylabel("Total Sales")
plt.tight_layout()
plt.show()

**Observations**

- The strongest monthly sales period is **2022-12**, while **2022-02** is the weakest month in the dataset.
- At the quarterly level, **2022Q4** leads total sales and **2022Q1** trails.
- The month-to-month pattern is not flat, which suggests promotional timing, seasonality, inventory availability, or customer demand cycles may be influencing revenue.

## 4. Customer Demographics Analysis

Customer demographics help identify the shopper groups contributing most transaction volume and revenue.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

age_order = ["18-24", "25-34", "35-44", "45-54", "55-64", "65+"]
sns.countplot(data=retail, x="age_group", order=age_order, ax=axes[0])
axes[0].set_title("Customer Age Group Distribution")
axes[0].set_xlabel("Age Group")
axes[0].set_ylabel("Transactions")

sns.countplot(data=retail, x="gender", order=retail["gender"].value_counts().index, ax=axes[1])
axes[1].set_title("Gender Breakdown")
axes[1].set_xlabel("Gender")
axes[1].set_ylabel("Transactions")

plt.tight_layout()
plt.show()

age_gender_revenue = retail.pivot_table(index="age_group", columns="gender", values="total_sale", aggfunc="sum", fill_value=0)
display(age_gender_revenue.round(0))

**Observations**

- The largest transaction age band is **45-54**, making it an important audience for retention and merchandising tests.
- The larger gender group by transaction count is **Female**.
- Looking at revenue by age group and gender together is more useful than counts alone, because a smaller segment can still generate high-value orders.

## 5. Product and Category Analysis

This dataset contains product categories rather than individual SKU names. Therefore, the top 10 best-selling products view is interpreted as the best-selling product/category groups available in the data.

In [ ]:
top_products = (
    retail.groupby("category", as_index=False)
    .agg(units_sold=("quantity", "sum"), revenue=("total_sale", "sum"))
    .sort_values(["units_sold", "revenue"], ascending=False)
    .head(10)
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.barplot(data=top_products, x="units_sold", y="category", hue="category", legend=False, ax=axes[0], palette="Blues_r")
axes[0].set_title("Top 10 Best-Selling Product Groups")
axes[0].set_xlabel("Units Sold")
axes[0].set_ylabel("Product / Category")

category_revenue = retail.groupby("category", as_index=False)["total_sale"].sum().sort_values("total_sale", ascending=False)
sns.barplot(data=category_revenue, x="total_sale", y="category", hue="category", legend=False, ax=axes[1], palette="Greens_r")
axes[1].set_title("Revenue by Product Category")
axes[1].set_xlabel("Total Revenue")
axes[1].set_ylabel("Category")

plt.tight_layout()
plt.show()

display(top_products)

**Observations**

- **Electronics** generates the highest category revenue at approximately **313,810**.
- Unit sales and revenue should be evaluated together: a category can sell many units but contribute less revenue if average selling price is lower.
- Category-level concentration means inventory, promotions, and display space should be prioritized around the leading revenue categories while still protecting variety.

## 6. Correlation Heatmap

The heatmap shows relationships between numerical fields such as age, quantity, price, cost, and total sale. Identifier columns are excluded because transaction/customer IDs do not represent measurable business magnitude.

In [ ]:
correlation_cols = ["age", "quantity", "price_per_unit", "cogs", "total_sale", "hour"]
corr = retail[correlation_cols].corr(numeric_only=True)

plt.figure(figsize=(9, 6))
sns.heatmap(corr, annot=True, cmap="vlag", center=0, linewidths=0.5, fmt=".2f")
plt.title("Correlation Matrix Between Numerical Variables")
plt.tight_layout()
plt.show()

**Observations**

- `total_sale` is mechanically related to both `quantity` and `price_per_unit`, so strong positive correlations are expected.
- `age` generally has a weaker relationship with transaction value than price and quantity, indicating demographics alone may not predict basket size.
- `cogs` should be monitored alongside revenue because high sales do not always imply high margin.

## 7. Additional Visualisation: Revenue by Hour and Category

This visualisation looks for a non-obvious operational insight: when customers buy, and whether category demand changes by time of day.

In [ ]:
hour_category = retail.pivot_table(index="hour", columns="category", values="total_sale", aggfunc="sum", fill_value=0)

plt.figure(figsize=(12, 7))
sns.heatmap(hour_category, cmap="YlGnBu", linewidths=0.4)
plt.title("Revenue Heatmap by Hour of Day and Category")
plt.xlabel("Category")
plt.ylabel("Hour of Day")
plt.tight_layout()
plt.show()

hourly_sales = retail.groupby("hour", as_index=False)["total_sale"].sum().sort_values("total_sale", ascending=False)
display(hourly_sales.head(10))

**Observations**

- The highest revenue hour is **19:00**, which is a useful clue for staffing, merchandising, and promotion timing.
- Category demand is not always evenly distributed across the day; hourly/category combinations with darker cells are candidates for targeted offers or operational focus.
- This view can uncover timing opportunities that would be hidden in a simple daily or monthly sales chart.

## 8. Conclusion and Actionable Recommendations

1. **Prioritize inventory and promotions around the strongest category.** The leading revenue category should receive reliable stock coverage, prominent placement, and carefully timed campaigns because it contributes the most top-line value.

2. **Use seasonality for campaign planning.** Since sales vary by month and quarter, plan larger promotions and inventory buffers before the strongest periods, while using weaker periods for discounting, bundles, or customer reactivation.

3. **Target the largest customer segments, but measure value not just volume.** The most common age and gender groups are important for reach, but revenue by segment should guide messaging, offers, and loyalty incentives.

4. **Optimize operations around peak selling hours.** The hourly revenue heatmap suggests staffing, checkout support, and time-bound promotions should align with the busiest revenue windows.

5. **Track margin alongside revenue.** Because COGS varies by transaction/category, high-revenue categories should be evaluated with profitability metrics before scaling discounts.

In [ ]:
# Optional export tables for reporting
monthly_sales.to_csv("monthly_sales_summary.csv", index=False)
quarterly_sales.to_csv("quarterly_sales_summary.csv", index=False)
category_revenue.to_csv("category_revenue_summary.csv", index=False)
top_products.to_csv("top_product_groups_summary.csv", index=False)